In [5]:
import pandas as pd
import re
import nltk
import joblib

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix
)

# =========================================================
# 1. DOWNLOAD NLTK RESOURCES
# =========================================================

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# =========================================================
# 2. TEXT CLEANING FUNCTION
# =========================================================

def clean_text(text):

    # Handle non-string safely
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenize
    words = word_tokenize(text)

    # Stopwords
    stop_words = set(stopwords.words('english'))

    # Keep important negation words
    stop_words.discard('not')
    stop_words.discard('no')

    # Remove stopwords
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

# =========================================================
# 3. LOAD DATASET
# =========================================================

print("Loading dataset...")

url = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/train.tsv"

# Read TSV file
df = pd.read_csv(
    url,
    sep='\t',
    header=None,
    names=['text', 'label']
)

print("\nDataset Loaded Successfully!")

# =========================================================
# 4. BASIC EXPLORATION
# =========================================================

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nClass Distribution:")
print(df['label'].value_counts())

# =========================================================
# 5. CLEAN TEXT
# =========================================================

print("\nCleaning text...")

df['cleaned_text'] = df['text'].apply(clean_text)

print("\nOriginal vs Cleaned:")
print(df[['text', 'cleaned_text']].head())

# =========================================================
# 6. FEATURES & TARGET
# =========================================================

X = df['cleaned_text']
y = df['label']

# =========================================================
# 7. TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTrain/Test Split Done!")

# =========================================================
# 8. TF-IDF VECTORIZATION
# =========================================================

print("\nVectorizing text...")

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Vectorization Complete!")

# =========================================================
# 9. TRAIN MODEL
# =========================================================

print("\nTraining Logistic Regression...")

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Training Complete!")

# =========================================================
# 10. EVALUATION
# =========================================================

print("\nMaking Predictions...")

predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print("\n========== RESULTS ==========")

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

# =========================================================
# 11. CUSTOM PREDICTION TEST
# =========================================================

print("\n========== CUSTOM TEST ==========")

sample_text = "This movie was absolutely fantastic and inspiring"

# Clean text
cleaned_sample = clean_text(sample_text)

# Vectorize
vectorized_sample = vectorizer.transform([cleaned_sample])

# Predict
prediction = model.predict(vectorized_sample)[0]

# Confidence
probabilities = model.predict_proba(vectorized_sample)

confidence = probabilities.max()

# Convert labels
label_map = {
    0: "Negative",
    1: "Positive"
}

print(f"\nInput: {sample_text}")
print(f"Cleaned: {cleaned_sample}")
print(f"Prediction: {label_map[prediction]}")
print(f"Confidence: {confidence:.4f}")

# =========================================================
# 12. SAVE MODEL
# =========================================================

print("\nSaving model and vectorizer...")

joblib.dump(model, "sentiment_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")

print("\nFiles Saved Successfully!")

print("- sentiment_model.pkl")
print("- vectorizer.pkl")

# =========================================================
# END
# =========================================================

print("\nPipeline Completed Successfully!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PMLS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PMLS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PMLS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading dataset...

Dataset Loaded Successfully!

First 5 Rows:
                                                text  label
0  a stirring , funny and finally transporting re...      1
1  apparently reassembled from the cutting room f...      0
2  they presume their audience wo n't sit still f...      0
3  this is a visually stunning rumination on love...      1
4  jonathan parker 's bartleby should have been t...      1

Dataset Shape:
(6920, 2)

Class Distribution:
label
1    3610
0    3310
Name: count, dtype: int64

Cleaning text...

Original vs Cleaned:
                                                text  \
0  a stirring , funny and finally transporting re...   
1  apparently reassembled from the cutting room f...   
2  they presume their audience wo n't sit still f...   
3  this is a visually stunning rumination on love...   
4  jonathan parker 's bartleby should have been t...   

                                        cleaned_text  
0  stirring funny finally transporting imagin

In [6]:
python app.py


SyntaxError: invalid syntax (2255720966.py, line 1)